In [ ]:
# START HERE
!pip install -q implicit threadpoolctl pyarrow scipy pandas numpy scikit-learn
# Necessary installs for the model

In [ ]:
# --- Exploratory Data Analysis: Inspecting the JSON Schema ---
import json
import glob
import os

# Update this path if your data is located elsewhere!
data_folder = '/Users/karthikraturi/Documents/MPD Playlist Prediction Spotify Project/spotify_million_playlist_dataset/data/'

# Grab just the first file
sample_file = glob.glob(os.path.join(data_folder, "*.json"))[0]

print(f"Inspecting file: {os.path.basename(sample_file)}\n")

with open(sample_file, 'r') as f:
    data = json.load(f)

    print("--- Top Level Info ---")
    print(f"Top-level keys: {list(data.keys())}")

    # Grab the first playlist
    first_playlist = data['playlists'][0]
    print("\n--- Playlist Level Columns ---")
    for key in first_playlist.keys():
        if key != 'tracks':
            print(f"- {key}: {type(first_playlist[key]).__name__} (i.e.: {first_playlist[key]})")

    # Grab the first track of the first playlist
    first_track = first_playlist['tracks'][0]
    print("\n--- Track Level Columns ---")
    for key in first_track.keys():
        print(f"- {key}: {type(first_track[key]).__name__} (i.e.: {first_track[key]})")

Inspecting file: mpd.slice.549000-549999.json

--- Top Level Info ---
Top-level keys: ['info', 'playlists']

--- Playlist Level Columns ---
- name: str (i.e.: Bob Dylan)
- collaborative: str (i.e.: false)
- pid: int (i.e.: 549000)
- modified_at: int (i.e.: 1454803200)
- num_tracks: int (i.e.: 75)
- num_albums: int (i.e.: 65)
- num_followers: int (i.e.: 1)
- num_edits: int (i.e.: 28)
- duration_ms: int (i.e.: 18425368)
- num_artists: int (i.e.: 39)

--- Track Level Columns ---
- pos: int (i.e.: 0)
- artist_name: str (i.e.: Bob Dylan)
- track_uri: str (i.e.: spotify:track:6QHYEZlm9wyfXfEM1vSu1P)
- artist_uri: str (i.e.: spotify:artist:74ASZWbe4lXaubB36ztrGX)
- track_name: str (i.e.: Boots of Spanish Leather)
- album_uri: str (i.e.: spotify:album:7DZeLXvr9eTVpyI1OlqtcS)
- duration_ms: int (i.e.: 277106)
- album_name: str (i.e.: The Times They Are A-Changin')


In [ ]:
# Module 0: MPD JSON to Parquet ETL Pipeline
import os
import json
import pandas as pd
import glob
import time

def compile_mpd_data(data_folder, max_files=100):
    print(f"Starting ETL: Processing {max_files} JSON files...")
    start_time = time.time()

    files = glob.glob(os.path.join(data_folder, "*.json"))
    if max_files:
        files = files[:max_files]

    interactions = []
    track_metadata = {}    # Dict to deduplicate tracks
    playlist_metadata = [] # List to store playlist info even with dupe names

    # 1. Single Pass Parsing
    for file_path in files:
        with open(file_path, 'r') as f:
            data = json.load(f)

            for playlist in data['playlists']:
                pid = playlist['pid']

                # Add to Playlist Metadata Bucket (For the UI)
                playlist_metadata.append({
                    'pid': pid,
                    'name': playlist.get('name', 'Unknown Playlist'),
                    'num_tracks': playlist.get('num_tracks', 0),
                    'num_followers': playlist.get('num_followers', 0)
                })

                for track in playlist['tracks']:
                    uri = track['track_uri']

                    # Add to Math Bucket
                    interactions.append({'pid': pid, 'track_uri': uri})

                    # Add to Track Metadata Bucket (Deduplicated)
                    if uri not in track_metadata:
                        track_metadata[uri] = {
                            'track_uri': uri,
                            'track_name': track.get('track_name', ''),
                            'artist_name': track.get('artist_name', '')
                        }

    # 2. Build the DataFrames
    print("Building DataFrames...")

    # Interactions (Math)
    df_interactions = pd.DataFrame(interactions)
    df_interactions['pid'] = pd.to_numeric(df_interactions['pid'], downcast='integer')
    df_interactions['track_uri'] = df_interactions['track_uri'].astype('category')

    # Track Metadata (UI)
    df_track_meta = pd.DataFrame(list(track_metadata.values()))

    # Playlist Metadata (UI)
    df_playlist_meta = pd.DataFrame(playlist_metadata)
    df_playlist_meta['pid'] = pd.to_numeric(df_playlist_meta['pid'], downcast='integer')

    # 3. Save to Parquet
    print("Compressing and Saving to Parquet...")
    df_interactions.to_parquet('mpd_interactions.parquet', engine='pyarrow', index=False)
    df_track_meta.to_parquet('mpd_track_metadata.parquet', engine='pyarrow', index=False)
    df_playlist_meta.to_parquet('mpd_playlist_metadata.parquet', engine='pyarrow', index=False)

    elapsed = time.time() - start_time
    print(f"\n=== ETL COMPLETE ({elapsed:.2f} seconds) ===")
    print(f"Interactions Shape: {df_interactions.shape[0]:,} rows")
    print(f"Unique Tracks Saved: {df_track_meta.shape[0]:,}")
    print(f"Playlists Saved: {df_playlist_meta.shape[0]:,}")
    print("\nGenerated Files:")
    print("1. mpd_interactions.parquet")
    print("2. mpd_track_metadata.parquet")
    print("3. mpd_playlist_metadata.parquet")

# --- RUN THE ETL ---
# Make sure your data_folder path is correct
data_folder = '/Users/karthikraturi/Documents/MPD Playlist Prediction Spotify Project/spotify_million_playlist_dataset/data/'

# Set max_files to None if you want to run the entire dataset!
compile_mpd_data(data_folder, max_files=100)

Starting ETL: Processing 100 JSON files...
Building DataFrames...
Compressing and Saving to Parquet...

=== ETL COMPLETE (17.30 seconds) ===
Interactions Shape: 6,615,180 rows
Unique Tracks Saved: 682,955
Playlists Saved: 100,000

Generated Files:
1. mpd_interactions.parquet
2. mpd_track_metadata.parquet
3. mpd_playlist_metadata.parquet


In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import scipy.sparse as sparse
import implicit
import time
import threadpoolctl
import gc
import joblib

In [ ]:
# Module 1: Sliced Data Loading and Matrix creation

# --- Force OpenBLAS to use 1 thread dynamically ---
threadpoolctl.threadpool_limits(1, "blas")


# MLOps Configuration
DEV_MODE = True
DEV_SAMPLE_SIZE = 100000  # Loads 10% of the data for fast prototyping



def load_and_build_sparse_matrix(interactions_path, dev_mode=True):
    start_time = time.time()

    # 1. Schema-on-Read Filtering (The DataOps Hack)
    if dev_mode:
        print(f"--- DEV MODE: Loading first {DEV_SAMPLE_SIZE:,} playlists ---")
        df = pd.read_parquet(
            interactions_path,
            columns=['pid', 'track_uri'],
            filters=[('pid', '<', DEV_SAMPLE_SIZE)]
        )
    else:
        print("--- PROD MODE: Loading full 1M dataset ---")
        df = pd.read_parquet(interactions_path, columns=['pid', 'track_uri'])

    print(f"Loaded {len(df):,} interactions. Converting to Sparse Matrix...")

    # 2. Categorical Encoding (Assign an Integer ID to every Track URI)
    df['track_uri'] = df['track_uri'].astype('category')

    # 3. Build the CSR Sparse Matrix
    # rows = playlists, cols = tracks, data = '1' (implicit positive feedback)
    playlists = df['pid'].values
    tracks = df['track_uri'].cat.codes.values
    data = np.ones(len(df), dtype=np.float32) # The implicit library prefers float32

    user_item_matrix = sparse.csr_matrix((data, (tracks, playlists)))

    # 4. Save the Translation Dictionary
    # We need to map the Integer IDs back to Spotify URIs later
    id_to_uri = dict(enumerate(df['track_uri'].cat.categories))
    uri_to_id = {uri: i for i, uri in id_to_uri.items()}

    # Free up RAM by deleting the DataFrame
    del df
    gc.collect()

    elapsed = time.time() - start_time
    print(f"Matrix Built! Shape: {user_item_matrix.shape[0]:,} Playlists x {user_item_matrix.shape[1]:,} Tracks")
    print(f"Data Prep Time: {elapsed:.2f} seconds\n")

    return user_item_matrix, id_to_uri, uri_to_id

In [ ]:
# Module 2: Train ALS Engine
def train_als_engine(user_item_matrix, factors=100, regularization=0.1, iterations=15):
    print(f"--- Training ALS Engine (Factors: {factors}, Reg: {regularization}) ---")
    start_time = time.time()

    # Initialize the Alternating Least Squares Model
    # random_state ensures reproducibility for our MLOps tracking
    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42,
        calculate_training_loss=True
    )

    # Fit the model using the sparse matrix
    # Under the hood, this uses low-level C++ and all your CPU cores
    model.fit(user_item_matrix)

    elapsed = time.time() - start_time
    print(f"Training Complete in {elapsed:.2f} seconds.\n")

    return model

# --- Execute Pipeline ---
interactions_file = 'mpd_interactions.parquet'

# 1. Build the Matrix
user_items, id_to_uri, uri_to_id = load_and_build_sparse_matrix(
    interactions_file,
    dev_mode=DEV_MODE
)

# 2. Train the Model
als_model = train_als_engine(
    user_item_matrix=user_items,
    factors=100,
    regularization=0.1,
    iterations=15
)


#==========================================================================================================================================



# Module 3: Native Track-to-Playlist Evaluation (MAP@K & NDCG)
from implicit.evaluation import train_test_split, mean_average_precision_at_k, ndcg_at_k
import implicit

def evaluate_als_model(track_playlist_matrix, factors=100, regularization=0.1, iterations=15, K=10):
    print("--- 1. Splitting Data (80% Train / 20% Hidden) ---")
    # Since your matrix is ALREADY Tracks x Playlists, we pass it straight in.
    # This hides 20% of the true playlists for a subset of tracks.
    train_matrix, test_matrix = train_test_split(track_playlist_matrix, train_percentage=0.8, random_state=42)

    print("--- 2. Training Strict Evaluation Model ---")
    # We MUST train a new model strictly on the 80% split to avoid data leakage.
    eval_model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42
    )
    eval_model.fit(train_matrix)

    print(f"--- 3. Calculating MAP@{K} and NDCG@{K} ---")
    # The library thinks it's predicting "Items for Users", but because of your matrix structure,
    # it is calculating the accuracy of predicting "Playlists for Tracks"!
    map_score = mean_average_precision_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)
    ndcg_score = ndcg_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)

    print("\n=== FINAL EVALUATION METRICS ===")
    print(f" MAP@{K}:  {map_score:.4f}")
    print(f" NDCG@{K}: {ndcg_score:.4f}\n")

    return map_score, ndcg_score

# --- RUN THE TEST ---
# Run this using the matrix you generated in Module 1
map_score, ndcg_score = evaluate_als_model(
    track_playlist_matrix=user_items,
    factors=100,
    regularization=0.1,
    iterations=15,
    K=10
)
# Module 3: Native Track-to-Playlist Evaluation (MAP@K & NDCG)
from implicit.evaluation import train_test_split, mean_average_precision_at_k, ndcg_at_k
import implicit

def evaluate_als_model(track_playlist_matrix, factors=100, regularization=0.1, iterations=15, K=10):
    print("--- 1. Splitting Data (80% Train / 20% Hidden) ---")
    # Since your matrix is ALREADY Tracks x Playlists, we pass it straight in.
    # This hides 20% of the true playlists for a subset of tracks.
    train_matrix, test_matrix = train_test_split(track_playlist_matrix, train_percentage=0.8, random_state=42)

    print("--- 2. Training Strict Evaluation Model ---")
    # We MUST train a new model strictly on the 80% split to avoid data leakage.
    eval_model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        regularization=regularization,
        iterations=iterations,
        random_state=42
    )
    eval_model.fit(train_matrix)

    print(f"--- 3. Calculating MAP@{K} and NDCG@{K} ---")
    # The library thinks it's predicting "Items for Users", but because of your matrix structure,
    # it is calculating the accuracy of predicting "Playlists for Tracks"!
    map_score = mean_average_precision_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)
    ndcg_score = ndcg_at_k(eval_model, train_matrix, test_matrix, K=K, show_progress=True)

    print("\n=== FINAL EVALUATION METRICS ===")
    print(f" MAP@{K}:  {map_score:.4f}")
    print(f" NDCG@{K}: {ndcg_score:.4f}\n")

    return map_score, ndcg_score

# --- RUN THE TEST ---
# Run this using the matrix you generated in Module 1
map_score, ndcg_score = evaluate_als_model(
    track_playlist_matrix=user_items,
    factors=100,
    regularization=0.1,
    iterations=15,
    K=10
)

--- DEV MODE: Loading first 100,000 playlists ---
Loaded 666,201 interactions. Converting to Sparse Matrix...
Matrix Built! Shape: 682,955 Playlists x 85,000 Tracks
Data Prep Time: 0.61 seconds

--- Training ALS Engine (Factors: 100, Reg: 0.1) ---


  0%|          | 0/15 [00:00<?, ?it/s]

Training Complete in 45.40 seconds.



In [ ]:
# Module 4: Metadata-Driven Popularity Baseline
import pandas as pd
import numpy as np
from implicit.evaluation import train_test_split
from tqdm.notebook import tqdm

def calculate_ndcg_at_k(recs, truth, k):
    dcg = 0.0
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(truth), k)))
    for i, rec in enumerate(recs[:k]):
        if rec in truth:
            dcg += 1.0 / np.log2(i + 2)
    return dcg / idcg if idcg > 0 else 0.0

def calculate_map_at_k(recs, truth, k):
    hits = 0
    sum_precs = 0.0
    for i, rec in enumerate(recs[:k]):
        if rec in truth:
            hits += 1
            sum_precs += hits / (i + 1.0)
    return sum_precs / min(len(truth), k) if len(truth) > 0 else 0.0

def evaluate_metadata_baseline(track_playlist_matrix, playlist_meta_path, K=10, num_test_tracks=5000):
    print("--- 1. Re-Splitting Data (Matching the ALS Eval) ---")
    train_matrix, test_matrix = train_test_split(track_playlist_matrix, train_percentage=0.8, random_state=42)

    print("--- 2. Loading Metadata for True Popularity ---")
    # Load the metadata file
    playlist_meta = pd.read_parquet(playlist_meta_path, columns=['pid', 'num_followers'])

    # Sort by followers to find the heavy hitters
    top_playlists_df = playlist_meta.sort_values(by='num_followers', ascending=False)

    # Extract the PIDs as a clean numpy array for fast iteration
    top_popular_pids = top_playlists_df['pid'].values

    print("--- 3. Prepping Test Data ---")
    test_tracks = np.where(test_matrix.getnnz(axis=1) > 0)[0]
    if len(test_tracks) > num_test_tracks:
        np.random.seed(42)
        test_tracks = np.random.choice(test_tracks, num_test_tracks, replace=False)

    pop_ndcg_scores, pop_map_scores = [], []

    print(f"--- 4. Evaluating Follower Baseline for {num_test_tracks} Tracks ---")
    for track_id in tqdm(test_tracks, desc="Testing Metadata Base"):
        true_playlists = set(test_matrix[track_id].indices)
        if not true_playlists: continue
        train_playlists = set(train_matrix[track_id].indices)

        # Build baseline recommendations: top Followed playlists excluding ones the track is already in
        pop_recs = []
        for pid in top_popular_pids:
            if pid not in train_playlists:
                pop_recs.append(pid)
            if len(pop_recs) == K:
                break

        # Calculate Metrics
        pop_ndcg_scores.append(calculate_ndcg_at_k(pop_recs, true_playlists, K))
        pop_map_scores.append(calculate_map_at_k(pop_recs, true_playlists, K))

    print("\n=== METADATA BASELINE METRICS (BY FOLLOWERS) ===")
    print(f"MAP@{K}:  {np.mean(pop_map_scores):.4f}")
    print(f"NDCG@{K}: {np.mean(pop_ndcg_scores):.4f}\n")

    return np.mean(pop_map_scores), np.mean(pop_ndcg_scores)

# --- RUN THE BASELINE ---
base_map, base_ndcg = evaluate_metadata_baseline(
    track_playlist_matrix=user_items,
    playlist_meta_path='mpd_playlist_metadata.parquet', # Point to your saved Parquet file
    K=10,
    num_test_tracks=5000
)

--- 1. Re-Splitting Data (Matching the ALS Eval) ---
--- 2. Loading Metadata for True Popularity ---
--- 3. Prepping Test Data ---
--- 4. Evaluating Follower Baseline for 5000 Tracks ---


Testing Metadata Base:   0%|          | 0/5000 [00:00<?, ?it/s]


=== METADATA BASELINE METRICS (BY FOLLOWERS) ===
MAP@10:  0.0001
NDCG@10: 0.0002



In [ ]:
# Module 4 Model Persistence (For use when training full dataset)
# Save the trained model and the dictionaries to disk
print("Saving model artifacts...")
joblib.dump(als_model, 'als_model_dev.joblib')
joblib.dump(uri_to_id, 'uri_to_id.joblib')
joblib.dump(id_to_uri, 'id_to_uri.joblib')
sparse.save_npz('user_items.npz', user_items)
print("Artifacts saved successfully!")

Saving model artifacts...
Artifacts saved successfully!


In [ ]:
# Module 4.5 To load them back later:
# 1. Load the trained model and mappings back into RAM (Takes ~1 second)
print("Loading model artifacts from disk...")
als_model = joblib.load('als_model_dev.joblib')
uri_to_id = joblib.load('uri_to_id.joblib')
id_to_uri = joblib.load('id_to_uri.joblib')
user_items = sparse.load_npz('user_items.npz')
print("Model restored successfully!")

Loading model artifacts from disk...
Model restored successfully!


In [ ]:
# Module 6: MLFlow Testing



In [ ]:
# Module 7: Inference Model (Cosine Similarity)
import pandas as pd
import numpy as np

def recommend_playlists_for_track(test_uri, model, uri_to_id, track_meta_path, playlist_meta_path, min_followers=100, n_candidates=5000, n_recs=10):

    if test_uri not in uri_to_id:
        return "Error: Track not found in training set."

    # --- Fetch Track Title ---
    track_meta = pd.read_parquet(track_meta_path, filters=[('track_uri', '==', test_uri)])
    track_name = track_meta['track_name'].iloc[0] if not track_meta.empty else "Unknown Track"
    artist_name = track_meta['artist_name'].iloc[0] if not track_meta.empty else "Unknown Artist"

    print(f"--- FINDING PLAYLIST FIT FOR: '{track_name}' by {artist_name} ---")

    track_id = uri_to_id[test_uri]

    # 1. The Math: Cosine Similarity
    # --- UPDATED: Tracks are now user_factors (Rows), Playlists are item_factors (Columns) ---
    track_factors = model.user_factors[track_id]
    playlist_factors = model.item_factors

    # A. Raw Dot Product
    dot_products = playlist_factors.dot(track_factors)

    # B. Magnitudes (L2 Norms)
    track_norm = np.linalg.norm(track_factors)
    playlist_norms = np.linalg.norm(playlist_factors, axis=1)

    # C. Cosine Similarity (-1 to 1)
    cosine_scores = dot_products / (track_norm * playlist_norms + 1e-9)

    # D. Convert to Percentage (0 to 100)
    percentage_scores = ((cosine_scores + 1) / 2) * 100

    # 2. Get a wide candidate pool based on the new PERCENTAGE
    top_candidate_pids = np.argsort(percentage_scores)[::-1][:n_candidates]

    # 3. Lazy Load Metadata for candidates
    playlist_meta = pd.read_parquet(
        playlist_meta_path,
        filters=[('pid', 'in', top_candidate_pids.tolist())],
        columns=['pid', 'name', 'num_tracks', 'num_followers']
    )

    # 4. Apply Business Filter (Reach)
    filtered_meta = playlist_meta[playlist_meta['num_followers'] >= min_followers].copy()

    # 5. Map the formatted percentages back to the filtered metadata
    score_dict = {
        pid: f"{score:.1f}%"
        for pid, score in zip(top_candidate_pids, percentage_scores[top_candidate_pids])
    }
    filtered_meta['Match_Percentage'] = filtered_meta['pid'].map(score_dict)

    # We also need a numeric column to sort the filtered results properly
    filtered_meta['Sort_Score'] = filtered_meta['pid'].map(
        {pid: score for pid, score in zip(top_candidate_pids, percentage_scores[top_candidate_pids])}
    )

    # 6. Sort and format final output
    leads_df = filtered_meta.sort_values(by='Sort_Score', ascending=False).head(n_recs)

    # Add a Rank column for intuitive UI
    leads_df.insert(0, 'Rank', range(1, len(leads_df) + 1))

    # Clean up column names and drop the hidden sort column
    return leads_df[['Rank', 'pid', 'name', 'num_followers', 'Match_Percentage']].rename(
        columns={'pid': 'PID', 'name': 'Playlist Name', 'num_followers': 'Followers'}
    ).reset_index(drop=True)

# --- RUN THE TEST ---
user_items = sparse.load_npz('user_items.npz')

# Find a track with strong signal for testing
# --- UPDATED: Use axis=1 to sum across columns (Playlists) and find the most popular Track row ---
track_counts = user_items.sum(axis=1).A1
popular_track_id = np.argmax(track_counts)
sample_uri = id_to_uri[popular_track_id]

results_df = recommend_playlists_for_track(
    test_uri=sample_uri,
    model=als_model,
    uri_to_id=uri_to_id,
    track_meta_path='mpd_track_metadata.parquet',
    playlist_meta_path='mpd_playlist_metadata.parquet',
    min_followers=10,
    n_candidates=5000, # Increased to ensure enough candidates survive the 10-follower filter
    n_recs=10
)

results_df

--- FINDING PLAYLIST FIT FOR: 'Closer' by The Chainsmokers ---


,Rank,PID,Playlist Name,Followers,Match_Percentage
0,1,7206,November 2016,44,72.6%
1,2,51335,Promise,10,66.8%
2,3,81291,my favorites,12,64.6%
3,4,70354,my songs,261,60.0%
4,5,51759,hypebeast,10,59.9%
5,6,22997,Favorite songs,27,57.5%
6,7,7215,TOP POP,15842,55.5%
7,8,51286,Smoke,10,54.3%
8,9,84595,Classical,11,54.0%
9,10,75206,Jhene aiko,27,53.7%
